In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the project's modules
import src.main as main
import src.build_dataset as build_dataset
import src.geo_utils as geo_utils
from src.utils.paths import PROCESSED_DATA_DIR

# 1. Load the datasets and the dataframe
years = [2016, 2018, 2020, 2022, 2024] # Adjust depending on the years you want to test
print("Loading data...")
datasets_all_years, extents_all_years, df = main.open_datasets(sat_data="aerial", years=years)
print("Data loaded successfully!")

2026-02-23 13:05:17.294687: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 13:05:17.635524: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-23 13:05:18.999410: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Loading data...
Reading dataset...
Loading 1 files from /home/abbatenicolas/data...


/home/abbatenicolas/miniconda3/envs/tf/lib/python3.9/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'rasterio' loading failed:
libnetcdf.so.19: cannot open shared object file: No such file or directory
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


Links without images (2016): 0 out of 449727
Remaining links for train/test (2016): 449727


In [ ]:
# 2. Set the configuration matching your main.py training setup
image_size = 128 * 3  # Based on main.py params
resizing_size = 128
nbands = 4
stacked_images = [1, 3] 
available_years = list(datasets_all_years.keys())

# Take a random sample of 50 rows to verify
sample_size = 50
df_sample = df.sample(sample_size, random_state=825)

# Create a directory for debugging outputs
save_dir = PROCESSED_DATA_DIR / "debug_ingestion_examples"
os.makedirs(save_dir, exist_ok=True)

# 3. Replicate the generation logic from `main.create_datasets`
for idx, row in df_sample.iterrows():
    # Retrieve identification and location data
    geoid = row.get("GEOID", "Unknown")
    polygon = row["geometry"]
    centroid = polygon.centroid
    coords = (centroid.x, centroid.y)
    
    # Pick a random year identically to get_mini_batch_data
    batch_year = random.choice(available_years)
    primary_dataset = datasets_all_years[batch_year]
    dataset_name = row[f"dataset_{batch_year}"]
    
    if pd.isna(dataset_name):
        print(f"Skipping Index {idx}: No dataset assignment for year {batch_year}")
        continue
        
    link_dataset = primary_dataset[dataset_name]
    
    try:
        # EXACT function used during training pipeline
        image, _ = geo_utils.stacked_image_from_census_tract(
            dataset=link_dataset,
            polygon=polygon,
            img_size=image_size,
            n_bands=nbands,
            stacked_images=stacked_images,
        )
        
        # Apply the exact validation and resizing used in training
        total_bands = nbands * len(stacked_images)
        target_shape = (total_bands, image_size, image_size)
        
        if image.shape != target_shape:
             print(f"Shape mismatch at Index {idx}: Got {image.shape}, Expected {target_shape}. Yielding blank image.")
             image = np.zeros(shape=(resizing_size, resizing_size, total_bands))
        else:
             image = geo_utils.process_image(image, resizing_size)
        
        # Plotting the image (Visualizing the first 3 bands as RGB for validation)
        plt.figure(figsize=(6, 6))
        
        # Depending on how `process_image` formats your array, 
        # ensure it's in (H, W, Channels) before plotting.
        # Assuming the output is already (128, 128, 8), we slice the first 3 channels:
        vis_image = image[:, :, :3]
        
        # Normalize for plotting if it isn't already 0-255 or 0-1
        if vis_image.max() > 1.0:
            vis_image = vis_image.astype(float) / 255.0
            
        plt.imshow(vis_image)
        
        # Document the Index, GEOID, and Coordinates in the plot
        title_text = (f"Index: {idx} | GEOID: {geoid}\n"
                      f"Coords: ({coords[0]:.4f}, {coords[1]:.4f})\n"
                      f"Year: {batch_year} | Dataset: {dataset_name}")
        plt.title(title_text, fontsize=10)
        plt.axis('off')
        
        # Save out to disk
        filename = save_dir / f"idx_{idx}_geoid_{geoid}.png"
        plt.savefig(filename, bbox_inches='tight')
        plt.show()
        plt.close()
        
        print(f"Successfully exported verification image for Index {idx} -> {filename}")
        
    except Exception as e:
        print(f"Failed to process Index {idx} (GEOID: {geoid}): {e}")

print("\nVerification extraction complete. Review the outputs to check for coordinate/bounding box shifts.")